In [47]:
import torch

In [1]:
import numpy as np

emb_path = r"C:\Users\Adi\Downloads\all_embeddings.npy"
prm_path = r"C:\Users\Adi\Downloads\all_parametric.npy"

In [32]:
params = np.load(prm_path)
og_embs = np.load(emb_path)
print("loaded data")

loaded data


In [ ]:
print(f"Embeddings shape: {og_embs.shape}")

Embeddings shape: (10, 512)


In [ ]:
print(f"First 5 CAD parameters: {params[0][:5]}")
print(f"First 5 CLIP embedding values: {og_embs[0][:5]}")

First 5 CAD parameters: [367.08961089  61.40294087 512.07647055  62.31507936 173.20703377]
First 5 CLIP embedding values: [ 0.03507308  0.12974662  0.21229585 -0.01336599 -0.07249392]


In [5]:
from sentence_transformers import SentenceTransformer, util
from PIL import Image

c:\miniconda3\envs\biked\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
#Load CLIP model
model = SentenceTransformer('clip-ViT-B-32')

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 1010.15it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: C:\Users\Adi\.cache\huggingface\hub\models--sentence-transformers--clip-ViT-B-32\snapshots\327ab6726d33c0e22f920c83f2ff9e4bd38ca37f\0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [9]:
text_embedding = model.encode(["A rugged mountain bike with thick tires for dirt trails"])



In [33]:
print(type(og_embs[0][0]))
print(type(text_embedding[0][0]))

<class 'numpy.float64'>
<class 'numpy.float32'>


In [34]:
text_embedding = text_embedding.astype("float32")
og_embs = og_embs.astype("float32")

In [35]:
print(type(og_embs[0][0]))
print(type(text_embedding[0][0]))

<class 'numpy.float32'>
<class 'numpy.float32'>


In [ ]:
print(f"Parameters shape: {params.shape}")
print(f"Original Embeddings shape: {og_embs.shape}")
embs = og_embs[0:1000000]   
print(f"Child Embeddings shape: {embs.shape}")

Parameters shape: (1398150, 96)
Original Embeddings shape: (1398150, 512)
Child Embeddings shape: (1000000, 512)


In [46]:
cos_scores = util.cos_sim(og_embs, text_embedding)

print(cos_scores)

tensor([[0.2747],
        [0.2857],
        [0.2743],
        ...,
        [0.2768],
        [0.2712],
        [0.2692]])


In [48]:
best_match_idx = torch.argmax(cos_scores).item()
best_score = cos_scores[best_match_idx].item() 
print(f" Best match found at index: {best_match_idx}")
print(f" Similarity Score: {best_score:.4f}")

 Best match found at index: 126943
 Similarity Score: 0.3050


In [49]:
winning_bike_params = params[best_match_idx]

print("\n Here are the first 10 parameters for your mountain bike:")
print(winning_bike_params[:10])


 Here are the first 10 parameters for your mountain bike:
[393.69919893  84.99951701 589.79083478  74.95577246 163.30495971
 152.2566413  401.78254492  70.76706882 772.42463082  52.4627564 ]


In [51]:
import pandas as pd

In [50]:
import bike_pipeline_claude as bp

In [63]:
df = pd.read_csv(r"Biked_Reference_Data\clip_sBIKED_processed.csv")
df = df.drop(columns="Unnamed: 0")
row1 = df.iloc[0].copy()

In [64]:
row1

CS textfield                         430.0
BB textfield                          67.0
Stack                                565.6
Head angle                            73.0
Head tube length textfield           135.6
                                     ...  
RIM_STYLE front OHCLASS: SPOKED        1.0
RIM_STYLE front OHCLASS: TRISPOKE      0.0
RIM_STYLE rear OHCLASS: DISC           0.0
RIM_STYLE rear OHCLASS: SPOKED         1.0
RIM_STYLE rear OHCLASS: TRISPOKE       0.0
Name: 0, Length: 96, dtype: float64

In [65]:
df.columns

Index(['CS textfield', 'BB textfield', 'Stack', 'Head angle',
       'Head tube length textfield', 'Seat stay junction0', 'Seat tube length',
       'Seat angle', 'DT Length', 'BB diameter', 'ttd', 'dtd', 'csd', 'ssd',
       'Chain stay position on BB', 'SSTopZOFFSET',
       'Head tube upper extension2', 'Seat tube extension2',
       'Head tube lower extension2', 'SEATSTAYbrdgshift', 'CHAINSTAYbrdgshift',
       'SEATSTAYbrdgdia1', 'CHAINSTAYbrdgdia1', 'SEATSTAYbrdgCheck',
       'CHAINSTAYbrdgCheck', 'Dropout spacing',
       'Wall thickness Bottom Bracket', 'Wall thickness Top tube',
       'Wall thickness Head tube', 'Wall thickness Down tube',
       'Wall thickness Chain stay', 'Wall thickness Seat stay',
       'Wall thickness Seat tube', 'ERD rear', 'Wheel width rear', 'BSD front',
       'Wheel width front', 'ERD front', 'BSD rear', 'Display AEROBARS',
       'BB length', 'Head tube diameter', 'Wheel cut', 'Seat tube diameter',
       'Front Fender include', 'Rear Fender inc

In [67]:
feature = pd.DataFrame(params[best_match_idx].reshape(1,-1), columns=df.columns)



In [69]:
seriesFeature = feature.iloc[0]

In [70]:
pipeline = bp.BikePipeline()
pipeline.row_to_svg(seriesFeature, "uin1")


[bcad] Injected 49 features → uin1.bcad
[svg]  Rendered → uin1.svg


WindowsPath('Biked_Reference_Data/output/svg/uin1.svg')

In [59]:
row1

Unnamed: 0                             1.0
CS textfield                         430.0
BB textfield                          67.0
Stack                                565.6
Head angle                            73.0
                                     ...  
RIM_STYLE front OHCLASS: SPOKED        1.0
RIM_STYLE front OHCLASS: TRISPOKE      0.0
RIM_STYLE rear OHCLASS: DISC           0.0
RIM_STYLE rear OHCLASS: SPOKED         1.0
RIM_STYLE rear OHCLASS: TRISPOKE       0.0
Name: 0, Length: 97, dtype: float64